# 01 — NCM Hierarchy Inference

**Purpose:** run the full officially released NCM XSum test set, decoding every cumulative hierarchy level. The default is all 11,334 NCM evaluation inputs, not a pilot subset. Reference alignment is verified before the NCM model is loaded.

**Persistence:** the SQLite table commits every article atomically. If Colab disconnects, rerun this notebook after notebook 00. Fully completed article IDs are skipped. Batch size is conservative for T4; automatic OOM handling splits the in-memory batch while preserving output equivalence and refuses silent example skips.

This notebook generates model outputs once. Later notebooks read these records; they do not call the NCM generator.

In [ ]:
# ============================================================
# NOTEBOOK 0.1 — NCM HIERARCHY INFERENCE
# Full setup, dependency installation, and inference
# ============================================================

import os
import sys
import subprocess
import zipfile
from pathlib import Path

# Avoid errors from a disconnected Google Drive working directory.
os.chdir("/content")

def run_command(command, *, cwd=None, description=None):
    if description:
        print(f"\n{'=' * 72}\n{description}\n{'=' * 72}", flush=True)

    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    if result.stdout:
        print(result.stdout, flush=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"{description or 'Command'} failed with exit code "
            f"{result.returncode}: {command!r}"
        )

    return result.stdout


print("Python:", sys.version, flush=True)
print("Executable:", sys.executable, flush=True)

# ------------------------------------------------------------
# 1. Mount Drive and resolve project paths
# ------------------------------------------------------------

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_PARENT = DRIVE_ROOT / "NCM Research"
PROJECT_ROOT = PROJECT_PARENT / "github"
HF_ROOT = PROJECT_PARENT / "huggingface"
ZIP_PATH = DRIVE_ROOT / "NCM_Research_Colab_Project.zip"

UPSTREAM_URL = "https://github.com/HazyResearch/ncm.git"
UPSTREAM_COMMIT = "c70274273f489abc73832b2b98584a8483b6b203"
UPSTREAM_ROOT = Path("/content/ncm-upstream")

PACKAGE_DIR = PROJECT_ROOT / "src" / "ncm_factuality"
REQUIREMENTS_FILE = PROJECT_ROOT / "requirements-colab.txt"

print("Project root:", PROJECT_ROOT)
print("Dataset repository root:", HF_ROOT)

# ------------------------------------------------------------
# 2. Restore project archive only if the package is missing
# ------------------------------------------------------------

if not PACKAGE_DIR.is_dir():
    if not ZIP_PATH.is_file():
        raise FileNotFoundError(
            f"Project package not found at {PACKAGE_DIR}, and archive "
            f"not found at {ZIP_PATH}."
        )

    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        drive_resolved = DRIVE_ROOT.resolve()

        for member in archive.infolist():
            name = member.filename
            target = (DRIVE_ROOT / name).resolve()

            if not (
                name == "NCM Research/"
                or name.startswith("NCM Research/")
            ):
                raise RuntimeError(f"Unexpected ZIP member: {name}")

            if target != drive_resolved and drive_resolved not in target.parents:
                raise RuntimeError(f"Unsafe ZIP member: {name}")

        archive.extractall(DRIVE_ROOT)

if not (PACKAGE_DIR / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing package initializer: {PACKAGE_DIR}")

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(f"Missing project metadata: {PROJECT_ROOT}")

if not REQUIREMENTS_FILE.is_file():
    raise FileNotFoundError(f"Missing requirements file: {REQUIREMENTS_FILE}")

# ------------------------------------------------------------
# 3. Get the exact upstream NCM revision
# ------------------------------------------------------------

if not (UPSTREAM_ROOT / ".git").is_dir():
    run_command(
        [
            "git", "clone", "--filter=blob:none",
            UPSTREAM_URL, str(UPSTREAM_ROOT),
        ],
        description="Cloning upstream NCM",
    )

run_command(
    ["git", "-C", str(UPSTREAM_ROOT), "fetch", "origin", UPSTREAM_COMMIT],
    description="Fetching pinned NCM revision",
)

run_command(
    ["git", "-C", str(UPSTREAM_ROOT), "checkout", "--detach", UPSTREAM_COMMIT],
    description="Checking out pinned NCM revision",
)

actual_commit = subprocess.check_output(
    ["git", "-C", str(UPSTREAM_ROOT), "rev-parse", "HEAD"],
    text=True,
).strip()

if actual_commit != UPSTREAM_COMMIT:
    raise RuntimeError(
        f"Incorrect NCM revision: {actual_commit}; expected {UPSTREAM_COMMIT}"
    )

print("Verified NCM commit:", actual_commit)

# ------------------------------------------------------------
# 4. Install dependencies WITHOUT upgrading pip/setuptools
#    and without switching NumPy versions repeatedly
# ------------------------------------------------------------

# The project's requirements specify numpy<2.0. Keep that
# constraint rather than overriding it with NumPy 2.x.
run_command(
    [
        sys.executable, "-m", "pip", "install",
        "-r", str(REQUIREMENTS_FILE),
    ],
    description="Installing project dependencies",
)

# Reinstall the project's declared NumPy version last, so its
# on-disk files are not left mixed between two NumPy releases.
run_command(
    [
        sys.executable, "-m", "pip", "install",
        "--force-reinstall", "--no-deps", "numpy==1.26.4",
    ],
    description="Reinstalling the project's pinned NumPy",
)

run_command(
    [
        sys.executable, "-m", "pip", "install",
        "-e", str(PROJECT_ROOT), "--no-deps",
    ],
    description="Installing the research package",
)

# ------------------------------------------------------------
# 5. Run all imports and inference in a fresh Python process.
#    This prevents stale NumPy modules in the notebook kernel
#    from being reused after the package installation.
# ------------------------------------------------------------

child_code = r'''
import os
import sys
import importlib
import importlib.util
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/NCM Research/github")
UPSTREAM_ROOT = Path("/content/ncm-upstream")
SRC_ROOT = PROJECT_ROOT / "src"
CONFIG_PATH = PROJECT_ROOT / "configs" / "experiment.yaml"

os.chdir(str(PROJECT_ROOT))

# Ensure the intended local project and pinned upstream checkout
# are the paths Python resolves.
sys.path.insert(0, str(UPSTREAM_ROOT))
sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

import numpy as np
print("NumPy:", np.__version__)
print("NumPy module path:", np.__file__)

if np.__version__ != "1.26.4":
    raise RuntimeError(
        "Unexpected NumPy version in fresh process: "
        + np.__version__
    )

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select a GPU runtime in Colab "
        "and rerun this cell."
    )

print("GPU:", torch.cuda.get_device_name(0))

spec = importlib.util.find_spec("ncm_factuality")
if spec is None:
    raise ModuleNotFoundError(
        f"Cannot import ncm_factuality from {SRC_ROOT}"
    )

print("Resolved project package:", spec.origin)

# Verify the previously failing import in this clean process.
from transformers import AutoTokenizer
print("Transformers tokenizer import: OK")

from ncm_factuality.config import load_config, resolve_paths
from ncm_factuality.input_data import prepare_inputs
from ncm_factuality.inference import run_full_inference

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing experiment config: {CONFIG_PATH}")

cfg = load_config(CONFIG_PATH)
paths = resolve_paths(PROJECT_ROOT, cfg)
paths["run_dir"].mkdir(parents=True, exist_ok=True)

print("Persistent run directory:", paths["run_dir"])

inputs = prepare_inputs(cfg, paths)
print("Reference alignment:", inputs["alignment_report"])

result = run_full_inference(
    cfg,
    paths,
    UPSTREAM_ROOT,
    inputs,
)

print("\\nInference result:")
print(result)
print("\\nNCM hierarchy inference completed.")
'''

run_command(
    [sys.executable, "-c", child_code],
    cwd="/content",
    description="Running NCM inference in a fresh Python process",
)

## Output contract

The completed run contains `research.sqlite3` (source of truth for resumable per-article outputs), `hierarchy_outputs.jsonl` (portable downstream input), `input_alignment.json`, and `experiment_manifest.json`. Raw source articles reside only under `private/source_articles.jsonl` on Drive and must not be copied into the public dataset repository.

If the run fails, read `inference_failure.json` and the stack trace. Correct the cause before resuming. Do not delete the SQLite file just to get past an error; completed article rows are the main compute-saving checkpoint.